# Chapter 1 · Session demo

**Goal:** see the deck run live: one `POST /v1/messages` call, what comes back, why it stopped, who owns the history, and what it costs.

- **Estimated API cost:** about $0.02-0.03 for one full run (default model `claude-sonnet-5-5`, mostly `effort="low"`).
- **Slides:** [presentation.html](../../presentations/01-api-fundamentals/presentation.html) · **Theory:** [README.md](README.md) · **Depth and exercises:** [01_practice.ipynb](01_practice.ipynb)
- Cells marked `# [live]` call the API; `# [offline]` cells are free. Run top to bottom.

## 0. Setup

Load the key from `.env`, create one client, and define three small helpers used in every section.

In [ ]:
import asyncio
import os
import time
from collections import Counter

import anthropic
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())                          # ANTHROPIC_API_KEY (+ optional CLAUDE_MODEL) from .env
assert os.getenv("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in .env"

MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-5-5")
client = anthropic.Anthropic()                      # reads the key from the environment; no network call yet

DEMO = {"model": MODEL, "max_tokens": 2048, "output_config": {"effort": "low"}}    # demo defaults
print("anthropic", anthropic.__version__, "| model:", MODEL)

In [ ]:
PRICES = {  # USD per million tokens (input, output), list prices from README 1.7
    "claude-fable-5-1": (10.00, 50.00),
    "claude-opus-5-5": (4.00, 20.00),
    "claude-sonnet-5-5": (2.00, 10.00),
    "claude-haiku-4-5-20251001": (1.00, 5.00),
}
assert MODEL in PRICES and not MODEL.startswith("claude-haiku"), \
    "This demo sends output_config.effort: set CLAUDE_MODEL to a 5.x model from PRICES"

def text_of(resp) -> str:
    """Join the text blocks. Filter by type, never resp.content[0].text."""
    return "".join(b.text for b in resp.content if b.type == "text")

def cost(usage, model: str = MODEL) -> float:
    """Uncached input + output only (cache reads and writes are priced differently)."""
    inp, out = PRICES[model]
    return (usage.input_tokens * inp + usage.output_tokens * out) / 1e6 # 10^6

def show(resp) -> None:
    u = resp.usage
    print("stop_reason:", resp.stop_reason)
    print("blocks:     ", [b.type for b in resp.content])
    print(f"usage:       input={u.input_tokens} output={u.output_tokens} (thinking included) ≈ ${cost(u):.5f}")
    print("text:       ", text_of(resp))

## 1. One endpoint, one call

Three fields are required: `model`, `max_tokens` (a ceiling that **includes thinking**) and `messages`. Everything else is an optional parameter on the same endpoint.
**Look at:** `stop_reason`, the list of block types, and the token counts.

In [ ]:
# [live] ~ $0.001
resp = client.messages.create(
    model=MODEL,
    max_tokens=2048,                                         # required: output ceiling, thinking included
    system="You are a concise assistant for a cloud architecture team.",   # optional: operator rules
    messages=[{"role": "user", "content": "In two sentences, what is an API gateway?"}],  # required
    output_config={"effort": "low"},                         # optional: usually less thinking (measure on your task)
)
show(resp)

## 2. The reply is a typed list, not a string

`resp.content` is a list of blocks (`thinking`, `text`, `tool_use`, ...). With adaptive thinking a `thinking` block can come **first**, and it has no `.text`.
**Look at:** why `content[0].text` crashes while `text_of()` (filter by type) always works.

In [ ]:
# [offline] A reply shaped like many current-model replies: thinking first, then text
from anthropic.types import Message

typical = Message.model_validate({
    "id": "msg_demo", 
    "type": "message", 
    "role": "assistant", 
    "model": MODEL,
    "content": [{"type": "thinking", "thinking": "", "signature": "sig"},
                {"type": "text", "text": "An API gateway is the single front door for your APIs."}],
    "stop_reason": "end_turn", 
    "stop_sequence": None, 
    "usage": {"input_tokens": 20, "output_tokens": 30},
})

print("live reply in section 1:", [b.type for b in resp.content], "(may change on the next call: don't rely on it)")
print("typical reply:          ", [b.type for b in typical.content])

try:
    typical.content[0].text
except AttributeError as e:
    print("content[0].text ->", type(e).__name__, "(a thinking block has no .text)")

print("text_of(...)    ->", text_of(typical))
print("text_of(resp)   ->", text_of(resp)[:80], "...")

## 3. Where instructions live

`system` carries operator rules (role, tone, format) and your code resends it on every call; the `user` turn carries the task. There is no `tool` role: tool results travel inside a `user` turn (section 5).
**Look at:** same question, two system prompts, two very different answers.

In [ ]:
# [live] 2 calls, ~ $0.002
question = [{"role": "user", "content": "What is a load balancer?"}]

for system in ("Explain it to a CFO in one sentence, no jargon.",
               "You are a terse senior SRE. Max 12 words, no full sentences."):
    r = client.messages.create(**DEMO, system=system, messages=question)
    print(f"system: {system}\n  -> {text_of(r)}\n")

## 4. Why it stopped: `max_tokens` and `stop_sequence`

`stop_reason` is the only reliable control signal. A tiny `max_tokens` **truncates** the answer (thinking can even eat the whole budget). A stop sequence ends the text right before the matched string.
**Look at:** `stop_reason`, the cut-off text, and `resp.stop_sequence`.

In [ ]:
# [live] 2 calls, ~ $0.002
explain = [{"role": "user", "content": "Explain TCP slow start in three sentences."}]
count = [{"role": "user", "content": "Count from 1 to 30, one number per line, nothing else."}]

cut = client.messages.create(**{**DEMO, "max_tokens": 20}, messages=explain)     # far too small on purpose
stopped = client.messages.create(**DEMO, stop_sequences=["7"], messages=count)

for label, r in (("max_tokens=20", cut), ('stop_sequences=["7"]', stopped)):
    print(f"{label:<21} stop_reason={r.stop_reason!r}  stop_sequence={r.stop_sequence!r}")
    print(f"{'':<21} blocks={[b.type for b in r.content]}  text={text_of(r).replace(chr(10), ' ')!r}\n")

## 5. Why it stopped: `tool_use`

Give the model a tool and it may stop to ask **you** to run it. Your code runs it, sends the result back as a `tool_result` block in a `user` turn, and calls again (Chapter 2 builds the full loop).
**Look at:** `stop_reason='tool_use'` on the first call, `end_turn` on the second.

In [ ]:
# [live] ~ $0.0015
TOOLS = [{
    "name": "get_order",
    "description": "Look up the shipping status of an order by its ID.",
    "input_schema": {"type": "object", "properties": {"order_id": {"type": "string"}}, "required": ["order_id"]},
}]

ask = [{"role": "user", "content": "Where is my order A17?"}]
tool_resp = client.messages.create(**DEMO, tools=TOOLS, messages=ask)

show(tool_resp)
calls = [b for b in tool_resp.content if b.type == "tool_use"]

for c in calls:
    print("tool call:  ", c.name, c.input, c.id)

In [ ]:
# [live] ~ $0.0015. Run the tool ourselves, send ALL results in ONE user turn, call again
ORDERS = {"A17": "shipped 2026-10-01 via DHL, ETA 2026-10-06"}

results = [{"type": "tool_result", "tool_use_id": c.id,
            "content": ORDERS.get(str(c.input.get("order_id", "")).upper(), "no such order")} for c in calls]

if results:
    final = client.messages.create(**DEMO, tools=TOOLS, messages=ask + [
        {"role": "assistant", "content": tool_resp.content},     # the full content list, not just text
        {"role": "user", "content": results},
    ])
    show(final)
else:
    final = tool_resp
    print("The model answered without calling the tool this time; stop_reason:", tool_resp.stop_reason)

## 6. One handler for all seven stop reasons

Branch on `stop_reason`, never on the prose. `pause_turn`, `refusal` and `model_context_window_exceeded` are hard to trigger cheaply, so they appear only in the handler.
**Look at:** each live response mapped to the next action your code should take.

In [ ]:
# [offline] Uses the live responses from sections 1, 4 and 5
def next_action(resp) -> str:
    match resp.stop_reason:
        case "end_turn":      return "done: return text_of(resp)"
        case "tool_use":      return "run every tool_use block, send all results, call again"
        case "max_tokens":    return "TRUNCATED: raise max_tokens or stream; never run a half-written tool call"
        case "stop_sequence": return f"your stop string {resp.stop_sequence!r} fired: app logic"
        case "pause_turn":    return "server tool paused: send the assistant turn back unchanged"
        case "refusal":       return "declined: check BEFORE reading content; explain or fall back"
        case "model_context_window_exceeded": return "window full: shrink the INPUT, not max_tokens"
        case other:           return f"unknown {other!r}: fail loudly"

for name, r in [("first call", resp), ("truncated", cut), ("stop string", stopped),
                ("tool request", tool_resp), ("after tool", final)]:
    print(f"{name:<13} {r.stop_reason:<14} -> {next_action(r)}")

## 7. Stateless: your app owns the history

The API remembers nothing between calls: no session ID. To continue a chat you resend every earlier turn, and append the assistant's full `resp.content`, not just its text.
**Look at:** the forgetful answer, the remembering answer, and the extra input tokens memory costs.

In [ ]:
# [live] 3 calls, ~ $0.003
SYSTEM = "You are a friendly travel assistant. Answer in at most 40 words."

turn1 = {"role": "user", "content": "Hi, I'm Aida. I'm vegetarian and I love hiking."}

r1 = client.messages.create(**DEMO, system=SYSTEM, messages=[turn1])

followup = {"role": "user", "content": "What's my name, and what should I eat after a hike tonight?"}
forgot = client.messages.create(**DEMO, system=SYSTEM, messages=[followup])          # history NOT sent

history = [turn1, {"role": "assistant", "content": r1.content}, followup]              # full history
remembered = client.messages.create(**DEMO, system=SYSTEM, messages=history)

print(f"WITHOUT history ({forgot.usage.input_tokens} input tokens):\n  {text_of(forgot)}\n")
print(f"WITH history    ({remembered.usage.input_tokens} input tokens):\n  {text_of(remembered)}")

## 8. Context window and tokens: count before, measure after

`count_tokens` is free and returns an estimate before you send; `resp.usage` is what you were billed. The Models API tells you the window size, so you never hardcode it.
**Look at:** estimate vs. billed input, and how tiny one request is next to a 1M window (bigger window, same rule: curate).

In [ ]:
# [live] ~ $0.001 (count_tokens and models.retrieve are free)
probe = {"system": "You are a concise assistant.",
         "messages": [{"role": "user", "content": "Explain idempotency keys in two sentences."}]}
before = client.messages.count_tokens(model=MODEL, **probe)       # before: estimate, free
after = client.messages.create(**DEMO, **probe)                  # after: billed usage
info = client.models.retrieve(MODEL)

print(f"count_tokens (before): {before.input_tokens} input tokens")
print(f"usage (after):         {after.usage.input_tokens} input, {after.usage.output_tokens} output (thinking included)")
print(f"{info.display_name}: context window {info.max_input_tokens:,} tokens, max output {info.max_tokens:,}")
print(f"this request used {before.input_tokens / info.max_input_tokens:.4%} of the window")

## 9. Cost: same workload, different model

Cost = input tokens × input price + output tokens × output price (thinking is billed as output). Same workload as the deck: 1,000 requests × (2,000 input + 500 output tokens).
**Look at:** the spread between tiers. (The 5.x models use a newer tokenizer that produces ~30% more tokens for the same text than Haiku 4.5, so Haiku's real gap is a bit wider than this table shows.)

In [ ]:
# [offline] List prices from README 1.7
for m, (inp, out) in PRICES.items():
    print(f"{m:<27} ${1000 * (2000 * inp + 500 * out) / 1e6:>6.2f} per 1,000 requests")

print(f"\nThe live call in section 8 cost ≈ ${cost(after.usage):.5f}")

## 10. Model and effort

`output_config={"effort": ...}` sets how much the model may think (`low` … `max`; Sonnet 5.5 defaults to `high`). It is a budget, not a price: the model decides how to use it.
**Look at:** in our test runs, `low` often skipped the `thinking` block and reasoned out loud in the visible answer, breaking the one-line format and sometimes using **more** output tokens; `high` moved the reasoning into `thinking` and kept the format. Your run may differ. Measure effort on your own task.

In [ ]:
# [live] 2 calls, ~ $0.006
puzzle = [{"role": "user", "content":
    "Six services A-F are deployed one per day, Monday to Saturday. E is deployed before C. "
    "A is deployed exactly two days after F. B is not deployed on Monday, Wednesday or Saturday. "
    "D is deployed on the day right after B. B is deployed before C. C is not deployed on Saturday. "
    "F is not deployed on Monday. Reply with one line only, e.g. Mon=X Tue=Y Wed=Z Thu=W Fri=V Sat=U."}]

ANSWER = "Mon=E Tue=B Wed=D Thu=F Fri=C Sat=A"                  # the only valid schedule

for effort in ("low", "high"):
    t0 = time.perf_counter()
    r = client.messages.create(model=MODEL, max_tokens=4000, output_config={"effort": effort}, messages=puzzle)
    secs, lines = time.perf_counter() - t0, text_of(r).strip().splitlines()
    print(f"effort={effort:<4} blocks={[b.type for b in r.content]}  output_tokens={r.usage.output_tokens}  latency={secs:.1f}s")
    print(f"            answer: {len(lines)} line(s), correct={ANSWER in text_of(r)}, starts: {lines[0][:60]!r}\n")

## 11. Same request, different answer

Each token is sampled, so identical requests can return different text. Don't compare against one golden string: check a **property** over several runs and report a **pass rate**.
**Look at:** how many distinct sentences one identical request returns, while the property (mentions rain) still passes.

In [ ]:
# [live] 3 calls, ~ $0.002
RUNS = 3
scene = [{"role": "user", "content": "Write one vivid sentence describing a rainy morning in Almaty."}]
texts = [text_of(client.messages.create(**DEMO, messages=scene)).strip() for _ in range(RUNS)]

print(f"{len(set(texts))} distinct answers from {RUNS} identical requests")

for t in texts:
    print(f"  rain={'rain' in t.lower()!s:<5} {t}")

For a classifier the property is "one valid label". **Look at:** the pass rate, not whether one run matched a golden string.

In [ ]:
# [live] 3 calls, ~ $0.001
LABELS = {"billing", "technical", "account", "other"}
CLASSIFY = "Classify the support ticket as billing, technical, account or other. Reply with one lowercase word."

ticket = [{"role": "user", "content": "I was charged twice this month and now I can't log in to download the invoice."}]

labels = [text_of(client.messages.create(**DEMO, system=CLASSIFY, messages=ticket)).strip().lower().strip(".")
          for _ in range(RUNS)]

passed = sum(label in LABELS for label in labels)

print(f"labels: {dict(Counter(labels))}")
print(f"property 'is one valid label': {passed}/{RUNS} pass ({passed / RUNS:.0%})")

`temperature=0` is **not** the fix: current models reject non-default sampling parameters (and it never guaranteed identical output).

In [ ]:
# [live] Free (a rejected request is not billed). SDK 1.x has no temperature keyword, so send it raw.
try:
    client.messages.create(**DEMO, messages=ticket, extra_body={"temperature": 0.0})
    print("accepted (this model still allows temperature)")
except anthropic.BadRequestError as e:
    print("400 BadRequestError:", e.body.get("error", {}).get("message", e) if isinstance(e.body, dict) else e)

## 12. Production access: streaming and async

Streaming prints text as it is generated (use it for long outputs or a large `max_tokens`). `AsyncAnthropic` keeps several requests in flight; a semaphore caps how many, to stay under your rate limit.
**Look at:** text arriving in chunks, then the sum of three call latencies vs. the wall-clock time they took together.

In [ ]:
# [live] ~ $0.002
with client.messages.stream(**DEMO, messages=[
        {"role": "user", "content": "In 3 short bullet points: why stream LLM responses?"}]) as stream:
    for chunk in stream.text_stream:
        print(chunk, end="", flush=True)

msg = stream.get_final_message()

print(f"\n\nstop_reason={msg.stop_reason}  output_tokens={msg.usage.output_tokens}")

In [ ]:
# [live] 3 calls in parallel, ~ $0.002. Same API; more requests in flight, not a faster single request.
DOCS = ["Payments API returned 502 for 14 minutes after a bad config push; rolled back.",
        "Logging cluster disk filled up; ingestion paused for one hour until it was expanded.",
        "An expired TLS certificate broke the partner webhook endpoint overnight."]
limit = asyncio.Semaphore(3)                                        # max requests in flight

async def summarize(aclient, doc: str) -> tuple[str, float]:
    async with limit:
        t0 = time.perf_counter()
        r = await aclient.messages.create(**DEMO, messages=[
            {"role": "user", "content": f"Summarize in at most 12 words:\n{doc}"}])
        return text_of(r), time.perf_counter() - t0

async with anthropic.AsyncAnthropic() as aclient:                   # top-level await works in Jupyter
    t0 = time.perf_counter()
    results = await asyncio.gather(*(summarize(aclient, d) for d in DOCS))
    wall = time.perf_counter() - t0
    
print(f"sum of individual latencies: {sum(s for _, s in results):.1f}s | wall clock: {wall:.1f}s")
for text, _ in results:
    print("-", text)

## Recap

- **Filter output by type**, never by position: `text_of(resp)`, not `resp.content[0].text`.
- **Branch on `stop_reason`**, never on prose: `max_tokens` means truncated, `tool_use` means run tools and call again.
- **The API is stateless:** your app resends the full history (and pays for it in input tokens).
- **Count before, measure after:** `count_tokens` is free; `usage` is the bill, and thinking is billed as output.
- **Choose model and effort with an eval**, and judge it by pass rates over several runs.

Next: [01_practice.ipynb](01_practice.ipynb) for every concept in depth (all seven stop reasons, context tricks, platforms, the `ChatSession` mini-project), then [02_homework.ipynb](02_homework.ipynb) to test yourself.